# Phase 3 — Data Preprocessing

## Uncertainty-Aware Material Selection using Machine Learning

### Objective
Prepare the cleaned materials dataset for machine learning while preventing
data leakage and preserving physically meaningful information.

### Target
- `K_VRH`: Voigt-Reuss-Hill bulk modulus (GPa)

In [1]:
import pandas as pd
df = pd.read_csv("../data/processed/clean_materials_data.csv")
df.head()

,nsites,volume,space_group,elastic_anisotropy,K_VRH
0,12,194.419802,124,0.030688,194.268884
1,5,61.987320,164,0.266910,175.449907
2,2,25.952539,221,0.756489,295.077545
3,4,76.721433,63,2.376805,49.130670
4,12,160.300999,62,0.196930,256.768081


In [2]:
df.shape

(1181, 5)

In [3]:
X = df.drop(columns=["K_VRH"])
y = df["K_VRH"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X.head()

X shape: (1181, 4)
y shape: (1181,)


,nsites,volume,space_group,elastic_anisotropy
0,12,194.419802,124,0.030688
1,5,61.987320,164,0.266910
2,2,25.952539,221,0.756489
3,4,76.721433,63,2.376805
4,12,160.300999,62,0.196930


In [4]:
from sklearn.model_selection import train_test_split

# First split: 80% train, 20% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Second split: temporary set -> 10% validation, 10% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (944, 4) (944,)
Validation: (118, 4) (118,)
Test: (119, 4) (119,)


In [5]:
split_summary = pd.DataFrame({
    "Train": y_train.describe(),
    "Validation": y_val.describe(),
    "Test": y_test.describe()
})

split_summary

,Train,Validation,Test
count,944.000000,118.000000,119.000000
mean,135.572754,140.721644,137.284246
std,72.874957,76.494141,69.696040
min,6.476135,8.735581,7.512215
25%,72.761006,81.706338,81.588649
50%,130.552843,132.053551,122.646585
75%,188.936806,193.826666,192.006402
max,435.661487,385.194240,336.958408


In [6]:
print("Unique space groups")
print("Train:", X_train["space_group"].nunique())
print("Validation:", X_val["space_group"].nunique())
print("Test:", X_test["space_group"].nunique())

unseen_val = set(X_val["space_group"]) - set(X_train["space_group"])
unseen_test = set(X_test["space_group"]) - set(X_train["space_group"])

print("\nUnseen in validation:", sorted(unseen_val))
print("Unseen in test:", sorted(unseen_test))

Unique space groups
Train: 84
Validation: 32
Test: 37

Unseen in validation: [72]
Unseen in test: [33, 96]


In [7]:
handle_unknown="ignore"

In [8]:
categorical_features = [
    "space_group"
]

numerical_features = [
    "nsites",
    "volume",
    "elastic_anisotropy"
]

print("Categorical:", categorical_features)
print("Numerical:", numerical_features)

Categorical: ['space_group']
Numerical: ['nsites', 'volume', 'elastic_anisotropy']


In [9]:
df.nlargest(
    10,
    "elastic_anisotropy"
)[
    [
        "nsites",
        "volume",
        "space_group",
        "elastic_anisotropy",
        "K_VRH"
    ]
]

,nsites,volume,space_group,elastic_anisotropy,K_VRH
935,16,168.486154,69,397.297866,118.462873
1012,24,255.161402,36,320.518305,117.410033
373,2,21.142600,166,284.045653,117.548006
1170,4,43.223810,194,217.148741,102.717241
845,4,41.192934,194,209.960365,121.962397
715,12,224.004589,140,23.177239,86.340970
907,24,360.102860,15,22.302000,43.115000
289,8,111.284273,223,17.392709,159.329995
865,5,150.826478,164,14.946006,33.280764
140,16,236.973584,225,14.659718,36.957368


In [10]:
df_raw = pd.read_csv(
    "../data/raw/elastic_tensor.csv",
    skiprows=1
)

In [11]:
df_raw.nlargest(
    10,
    "elastic_anisotropy"
)[
    [
        "material_id",
        "formula",
        "space_group",
        "elastic_anisotropy",
        "K_VRH",
        "G_VRH"
    ]
]

,material_id,formula,space_group,elastic_anisotropy,K_VRH,G_VRH
935,mp-568286,C,69,397.297866,118.462873,91.024121
1012,mp-606949,C,36,320.518305,117.410033,91.339197
373,mp-169,C,166,284.045653,117.548006,90.237642
1170,mp-984,BN,194,217.148741,102.717241,74.521500
845,mp-48,C,194,209.960365,121.962397,95.525840
715,mp-30562,Sc2Co,140,23.177239,86.340970,12.336484
907,mp-557993,BiO2,15,22.302000,43.115000,24.590000
289,mp-1387,AlV3,223,17.392709,159.329995,10.578211
865,mp-505825,Cs2PtC2,164,14.946006,33.280764,14.424360
140,mp-11489,Li3Pd,225,14.659718,36.957368,11.819117


### Extreme Elastic Anisotropy Investigation

The elastic anisotropy distribution contains a small number of extreme
observations (>200). Inspection of the original material records showed that
four of the five most extreme observations correspond to carbon structures,
while one corresponds to BN.

These observations represent distinct material entries and are not removed
solely based on their statistical extremeness. They are retained to preserve
potentially meaningful physical behavior and will be considered explicitly
during later uncertainty analysis.

In [12]:
X_train[numerical_features].skew().sort_values(ascending=False)

elastic_anisotropy    14.711528
nsites                 4.419024
volume                 4.065470
dtype: float64

In [13]:
import numpy as np

X_train_log = X_train[numerical_features].copy()

for col in numerical_features:
    X_train_log[col] = np.log1p(X_train_log[col])

X_train_log.skew().sort_values(ascending=False)

elastic_anisotropy    4.027467
nsites                0.083492
volume               -0.186415
dtype: float64

### Numerical Transformation Decision

The numerical features exhibit substantial positive skewness in the training
data. A `log1p` transformation was evaluated without modifying the original
features.

The transformation substantially reduced skewness:

- `nsites`: 4.42 → 0.08
- `volume`: 4.07 → -0.19
- `elastic_anisotropy`: 14.71 → 4.03

Therefore, `log1p` transformation will be incorporated into the preprocessing
pipeline for the numerical features. Extreme anisotropy observations are
retained rather than removed.

In [14]:
X_train_log.describe().loc[["mean", "std", "min", "max"]]

,nsites,volume,elastic_anisotropy
mean,2.342582,5.029166,0.490791
std,0.713283,0.825800,0.611110
min,1.098612,2.916933,0.000005
max,5.030438,7.783185,5.987200


In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_log),
    columns=X_train_log.columns,
    index=X_train_log.index
)

X_train_scaled.describe().loc[["mean", "std", "min", "max"]]

,nsites,volume,elastic_anisotropy
mean,-1.373666e-16,-7.150589e-16,-7.526936e-17
std,1.000530e+00,1.000530e+00,1.000530e+00
min,-1.744932e+00,-2.559157e+00,-8.035310e-01
max,3.770288e+00,3.336738e+00,8.998903e+00


In [16]:
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    StandardScaler,
    OneHotEncoder
)

# Numerical preprocessing:
# log1p transformation -> standardization
numerical_pipeline = Pipeline([
    ("log_transform", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing:
# space_group is treated as a categorical crystallographic identifier
categorical_pipeline = Pipeline([
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

# Combine both preprocessing branches
preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

In [17]:
# Fit preprocessing only on the training data
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the learned preprocessing to validation and test data
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Train processed:", X_train_processed.shape)
print("Validation processed:", X_val_processed.shape)
print("Test processed:", X_test_processed.shape)

Train processed: (944, 87)
Validation processed: (118, 87)
Test processed: (119, 87)


In [18]:
feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(feature_names))
print("\nFirst 15 features:")

for feature in feature_names[:15]:
    print(feature)

Number of processed features: 87

First 15 features:
num__nsites
num__volume
num__elastic_anisotropy
cat__space_group_4
cat__space_group_8
cat__space_group_10
cat__space_group_11
cat__space_group_12
cat__space_group_13
cat__space_group_14
cat__space_group_15
cat__space_group_36
cat__space_group_38
cat__space_group_43
cat__space_group_46


In [19]:
print("NaN values:")
print("Train:", np.isnan(X_train_processed).sum())
print("Validation:", np.isnan(X_val_processed).sum())
print("Test:", np.isnan(X_test_processed).sum())

print("\nInfinite values:")
print("Train:", np.isinf(X_train_processed).sum())
print("Validation:", np.isinf(X_val_processed).sum())
print("Test:", np.isinf(X_test_processed).sum())

NaN values:
Train: 0
Validation: 0
Test: 0

Infinite values:
Train: 0
Validation: 0
Test: 0


## Preprocessing Summary

- The target (`K_VRH`) was separated from the four input features.
- The dataset was split into training (80%), validation (10%), and test (10%)
  sets using `random_state=42`.
- Split distributions were checked and no substantial target distribution
  shift was observed.
- `space_group` was treated as a categorical crystallographic identifier
  rather than a continuous numerical variable.
- Unseen space groups were identified in the validation and test sets.
  `OneHotEncoder(handle_unknown="ignore")` was therefore used.
- Numerical features were defined as `nsites`, `volume`, and
  `elastic_anisotropy`.
- Extreme elastic anisotropy observations were investigated using the
  original material records and retained rather than automatically removed.
- Numerical features showed substantial positive skewness. A `log1p`
  transformation substantially reduced this skewness and was incorporated
  into the numerical preprocessing pipeline.
- Numerical features were standardized using `StandardScaler`.
- All preprocessing parameters were learned exclusively from the training
  data to prevent data leakage.
- The final preprocessing pipeline produces 87 model-ready features:
  3 transformed numerical features and 84 one-hot encoded space-group
  features.
- No missing or infinite values were introduced during preprocessing.